In [ ]:
import wandb
from pathlib import Path
from dataclasses import dataclass, asdict

class WandbCallback:
    def __init__(self, log_every=50):
        self.log_every = log_every
        self.iteration = 0
        
    def __call__(self, env):
        # This gets called after each iteration
        if self.iteration % self.log_every == 0:
            # Log metrics to wandb
            metrics = {}
            for dataset_name, eval_name, value, _ in env.evaluation_result_list:
                metric_name = f"{dataset_name}/{eval_name}"
                metrics[metric_name] = value
            
            wandb.log(metrics, step=self.iteration)
        
        self.iteration += 1
        return False
    
@dataclass
class CFG:
    train_path: Path = Path("./data/train.csv")
    test_path: Path = Path("./data/test.csv")
    sub_path: Path = Path("./data/sample_submission.csv")

    num_fold: int = 5
    dev_mode: bool = False

    # Model parameters
    n_iter: int = 10000
    max_depth: int = -1
    num_leaves: int = 1024
    colsample_bytree: float = 0.7
    learning_rate: float = 0.03

    objective: str = 'l2'
    metric: str = 'rmse'
    verbosity: int = -1
    
    random_state: int = 42
    shuffle: bool = True
    encoded_columns_start: int = -91
    log_eval: int = 100
    early_stopping: int = 200
    
cfg = CFG() 
asdict(cfg)

{'train_path': PosixPath('data/train.csv'),
 'test_path': PosixPath('data/test.csv'),
 'sub_path': PosixPath('data/sample_submission.csv'),
 'num_fold': 5,
 'dev_mode': False,
 'n_iter': 10000,
 'max_depth': -1,
 'num_leaves': 1024,
 'colsample_bytree': 0.7,
 'learning_rate': 0.03,
 'objective': 'l2',
 'metric': 'rmse',
 'verbosity': -1,
 'max_bin': 1024,
 'random_state': 42,
 'shuffle': True,
 'encoded_columns_start': -91,
 'log_eval': 100,
 'early_stopping': 200}

In [35]:
from IPython.display import display
import pandas as pd
import numpy as np
from sklearn.metrics import mean_squared_error
import warnings
warnings.filterwarnings('ignore')
warnings.simplefilter('ignore')

def calc_rmse(y_true, y_pred):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    return rmse

re_dict = {}
re_dict['podc_dict'] = {
    'Mystery Matters': 0, 'Joke Junction': 1, 'Study Sessions': 2, 'Digital Digest': 3, 
    'Mind & Body': 4, 'Fitness First': 5, 'Criminal Minds': 6, 'News Roundup': 7, 
    'Daily Digest': 8, 'Music Matters': 9, 'Sports Central': 10, 'Melody Mix': 11, 
    'Game Day': 12, 'Gadget Geek': 13, 'Global News': 14, 'Tech Talks': 15, 
    'Sport Spot': 16, 'Funny Folks': 17, 'Sports Weekly': 18, 'Business Briefs': 19, 
    'Tech Trends': 20, 'Innovators': 21, 'Health Hour': 22, 'Comedy Corner': 23, 
    'Sound Waves': 24, 'Brain Boost': 25, "Athlete's Arena": 26, 'Wellness Wave': 27, 
    'Style Guide': 28, 'World Watch': 29, 'Humor Hub': 30, 'Money Matters': 31, 
    'Healthy Living': 32, 'Home & Living': 33, 'Educational Nuggets': 34, 
    'Market Masters': 35, 'Learning Lab': 36, 'Lifestyle Lounge': 37, 
    'Crime Chronicles': 38, 'Detective Diaries': 39, 'Life Lessons': 40, 
    'Current Affairs': 41, 'Finance Focus': 42, 'Laugh Line': 43, 
    'True Crime Stories': 44, 'Business Insights': 45, 'Fashion Forward': 46, 'Tune Time': 47
}
re_dict['genr_dict'] = {'True Crime': 0, 'Comedy': 1, 'Education': 2, 'Technology': 3, 'Health': 4, 'News': 5, 'Music': 6, 'Sports': 7, 'Business': 8, 'Lifestyle': 9}
re_dict['week_dict'] = {'Monday': 0, 'Tuesday': 1, 'Wednesday': 2, 'Thursday': 3, 'Friday': 4, 'Saturday': 5, 'Sunday': 6}
re_dict['time_dict'] = {'Morning': 10, 'Afternoon': 14, 'Evening': 17, 'Night': 21}
re_dict['sent_dict'] = {'Negative': 0, 'Neutral': 1, 'Positive': 2}


def preprocess(df):
    df['Episode_Num'] = df['Episode_Title'].str[8:].astype(int)  # Convert to int before log transform
    df = df.drop(columns=['Episode_Title'])

    # Convert categorical variables
    df['Genre'] = df['Genre'].replace(re_dict["genr_dict"])
    df['Podcast_Name'] = df['Podcast_Name'].replace(re_dict["podc_dict"])
    df['Publication_Day'] = df['Publication_Day'].replace(re_dict["week_dict"])
    df['Publication_Time'] = df['Publication_Time'].replace(re_dict["time_dict"])
    df['Episode_Sentiment'] = df['Episode_Sentiment'].replace(re_dict["sent_dict"])

    # Define categorical columns
    df["Episode_Length_minutes_NaN"] = df["Episode_Length_minutes"].isna().astype(int)
    df["Guest_Popularity_percentage_NaN"] = df["Guest_Popularity_percentage"].isna().astype(int)

    # Replacing null values by median
    df['Episode_Length_minutes'].fillna(df['Episode_Length_minutes'].median(), inplace=True)
    df['Guest_Popularity_percentage'].fillna(df['Guest_Popularity_percentage'].median(), inplace=True)
    
    return df


df_train = pd.read_csv(cfg.train_path, index_col='id')
df_test = pd.read_csv(cfg.test_path, index_col='id')
df_sub = pd.read_csv(cfg.sub_path, index_col='id')

# is_dev_mode = False
# # is_dev_mode = True
# if is_dev_mode:
#     df_train = df_train.sample(10000, random_state=42)
#     df_test = df_test[:10]
#     df_sub = df_sub[:10]
    
df_train = preprocess(df_train)
df_test = preprocess(df_test)

df_train_desc = df_train.describe()

# target_col = "Listening_Time_minutes"
# y_train = df_train[target_col].copy()
# df_train = df_train.drop(columns=[target_col])

display(df_train)
display(df_train_desc)

,Podcast_Name,Episode_Length_minutes,Genre,Host_Popularity_percentage,Publication_Day,Publication_Time,Guest_Popularity_percentage,Number_of_Ads,Episode_Sentiment,Listening_Time_minutes,Episode_Num,Episode_Length_minutes_NaN,Guest_Popularity_percentage_NaN
id,,,,,,,,,,,,,
0,0,63.84,0,74.81,3,21,53.58,0.0,2,31.41998,98,1,1
1,1,119.80,1,66.95,5,14,75.95,2.0,0,88.01241,26,0,0
2,2,73.90,2,69.97,1,17,8.97,0.0,0,44.92531,16,0,0
3,3,67.17,3,57.22,0,10,78.70,2.0,2,46.27824,45,0,0
4,4,110.51,4,80.07,0,14,58.68,3.0,1,75.61031,86,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
749995,36,75.66,2,69.36,5,10,53.58,0.0,0,56.87058,25,0,1
749996,19,75.75,8,35.21,5,21,53.58,2.0,1,45.46242,21,0,1
749997,37,30.98,9,78.58,3,10,84.89,0.0,0,15.26000,51,0,0


,Podcast_Name,Episode_Length_minutes,Genre,Host_Popularity_percentage,Publication_Day,Publication_Time,Guest_Popularity_percentage,Number_of_Ads,Episode_Sentiment,Listening_Time_minutes,Episode_Num,Episode_Length_minutes_NaN,Guest_Popularity_percentage_NaN
count,750000.000000,750000.000000,750000.000000,750000.000000,750000.000000,750000.000000,750000.000000,749999.000000,750000.000000,750000.000000,750000.000000,750000.000000,750000.000000
mean,23.540307,64.427546,4.556036,59.859901,3.030805,15.671500,52.498047,1.348855,0.997969,45.437406,51.445811,0.116124,0.194707
std,13.917884,30.996996,2.965912,22.873098,2.024196,4.026379,25.537152,1.151130,0.815440,27.138306,28.085623,0.320374,0.395975
min,0.000000,0.000000,0.000000,1.300000,0.000000,10.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000
25%,12.000000,39.420000,2.000000,39.410000,1.000000,14.000000,34.550000,0.000000,0.000000,23.178350,28.000000,0.000000,0.000000
50%,23.000000,63.840000,5.000000,60.050000,3.000000,17.000000,53.580000,1.000000,1.000000,43.379460,52.000000,0.000000,0.000000
75%,36.000000,90.310000,7.000000,79.530000,5.000000,21.000000,71.040000,2.000000,2.000000,64.811580,75.000000,0.000000,0.000000
max,47.000000,325.240000,9.000000,119.460000,6.000000,21.000000,119.910000,103.910000,2.000000,119.970000,100.000000,1.000000,1.000000


In [ ]:
df_over_ads = df_train[df_train["Number_of_Ads"] > 3]
display(df_over_ads)

print("RMSE score:", calc_rmse(df_over_ads["Listening_Time_minutes"], df_over_ads["Number_of_Ads"]))

,Podcast_Name,Episode_Length_minutes,Genre,Host_Popularity_percentage,Publication_Day,Publication_Time,Guest_Popularity_percentage,Number_of_Ads,Episode_Sentiment,Listening_Time_minutes,Episode_Num,Episode_Length_minutes_NaN,Guest_Popularity_percentage_NaN
id,,,,,,,,,,,,,
211159,27,64.83,4,48.46,6,17,53.58,53.37,2,50.44892,83,0,1
283606,22,109.93,4,67.81,6,10,77.90,103.91,1,103.89696,15,0,0
436577,10,115.25,7,28.58,5,14,23.65,103.00,1,103.12686,64,0,0
495919,23,64.83,1,48.37,5,17,53.58,53.42,1,50.44892,79,0,1
537705,24,112.27,6,28.95,5,10,10.15,103.75,1,103.12686,64,0,0
567235,39,16.13,0,49.11,1,17,43.17,12.00,2,6.49000,21,0,0
602553,24,112.27,6,28.19,1,10,23.15,103.25,1,103.12686,53,0,0
672139,24,115.74,6,28.95,1,14,23.50,103.25,1,103.12686,35,0,0
683147,38,109.93,0,59.66,1,17,77.91,103.88,1,103.89696,15,0,0


2.313134147390121

In [29]:
df_train_desc.loc["50%", "Episode_Length_minutes"]

63.84

In [24]:
from sklearn.model_selection import train_test_split
import lightgbm as lgb
import gc
import wandb
import os
from dotenv import load_dotenv

load_dotenv()
wandb.login(key=os.getenv("WANDB_API_KEY"))
wandb.init(project="playground-series-s5e4", config=asdict(cfg))

X = df_train.copy()
y = y_train.copy()
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, 
    test_size=0.2,  # 20% for validation
    random_state=42
)

X_test = df_test[X.columns].copy()

# # Target encoding if needed
# encoded_columns = df_train.columns[cfg.encoded_columns_start:]
# encoder = TargetEncoder(random_state=cfg.random_state)

# X_train[encoded_columns] = encoder.fit_transform(X_train[encoded_columns], y_train)
# X_valid[encoded_columns] = encoder.transform(X_valid[encoded_columns])
# X_test[encoded_columns] = encoder.transform(X_test[encoded_columns])

# Initialize the model
model = lgb.LGBMRegressor(
    n_iter=cfg.n_iter,
    max_depth=cfg.max_depth,
    num_leaves=cfg.num_leaves,
    colsample_bytree=cfg.colsample_bytree,
    learning_rate=cfg.learning_rate,
    objective=cfg.objective,
    metric=cfg.metric, 
    verbosity=cfg.verbosity,
    random_state=42,
)

# Train model with validation
model.fit(
    X_train, y_train,
    eval_set=[(X_train, y_train), (X_valid, y_valid)],
    callbacks=[
        lgb.log_evaluation(cfg.log_eval), 
        lgb.early_stopping(cfg.early_stopping),
        WandbCallback(log_every=10)
    ],
)

# Calculate validation score
val_score = model.best_score_['valid_1'][cfg.metric]
print(f"Validation score: {val_score}")
wandb.finish()
gc.collect()

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: Appending key for api.wandb.ai to your netrc file: /Users/masaishi/.netrc


Training until validation scores don't improve for 200 rounds
[100]	training's rmse: 12.6882	valid_1's rmse: 13.3882
[200]	training's rmse: 11.611	valid_1's rmse: 12.8348
[300]	training's rmse: 11.1507	valid_1's rmse: 12.776
[400]	training's rmse: 10.8097	valid_1's rmse: 12.7476
[500]	training's rmse: 10.5417	valid_1's rmse: 12.726
[600]	training's rmse: 10.2653	valid_1's rmse: 12.709
[700]	training's rmse: 10.0114	valid_1's rmse: 12.6945
[800]	training's rmse: 9.78491	valid_1's rmse: 12.679
[900]	training's rmse: 9.57247	valid_1's rmse: 12.6683
[1000]	training's rmse: 9.36269	valid_1's rmse: 12.6593
[1100]	training's rmse: 9.16221	valid_1's rmse: 12.6498
[1200]	training's rmse: 8.96751	valid_1's rmse: 12.6419
[1300]	training's rmse: 8.79177	valid_1's rmse: 12.6346
[1400]	training's rmse: 8.61922	valid_1's rmse: 12.629
[1500]	training's rmse: 8.44718	valid_1's rmse: 12.6238
[1600]	training's rmse: 8.28675	valid_1's rmse: 12.6204
[1700]	training's rmse: 8.10656	valid_1's rmse: 12.6185
[

wandb: ERROR The nbformat package was not found. It is required to save notebook history.


Validation score: 12.60973206519213


training/rmse,█▃▃▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
valid_1/rmse,█▄▄▃▃▃▃▃▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
training/rmse,6.89219
valid_1/rmse,12.61123


1648

## Median
Early stopping, best iteration is:
[2919]	training's rmse: 6.78114	valid_1's rmse: 12.6079
Validation score: 12.607863325639219

## no Median
[2896]	training's rmse: 6.45863	valid_1's rmse: 12.6143
wandb: ERROR The nbformat package was not found. It is required to save notebook history.
Validation score: 12.61426788237413

## Median with NaN Cols
Early stopping, best iteration is:
[2376]	training's rmse: 7.13339	valid_1's rmse: 12.6097
wandb: ERROR The nbformat package was not found. It is required to save notebook history.
Validation score: 12.60973206519213